<div align="right"><a href="L13-Entwurf-rekursiver-Filter.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). Please note that for running MATLAB in Notebooks, you have to install the [MATLAB API for Python](https://mathworks.com/help/matlab/matlab-engine-for-python.html) first. The installation process is described [here](https://mathworks.com/help/matlab/matlab_external/install-the-matlab-engine-for-python.html). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/matlab/S13-Designing-recursive-filters.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S12 Filter structures](./S12-Filter-Structures.ipynb) | [Exercise 13](./E13-Designing-recursive-filters.ipynb) | [Contents / Overview](./00-MATLAB-Notebooks-Overview-EN.ipynb) | Next: [S14 Quantization of filter coefficients](./S14-Quantization-of-filter-coefficients.ipynb)

</div>

---


# Exercise 4.2: Design of Recursive Filters

The goal of this exercise is to design a digital lowpass system based on a given tolerance scheme for its amplitude response. There are MATLAB functions e.g.~`butter`, `cheby1` etc. to realize a lowpass design for continuous time systems. The tolerance schema, that is given in the Z domain has to be transformed to the S domain. The known algorithms are used to design the desired filter in the S domain. Then the system function is transformed back to the Z domain.

To transform from Z- to S-domain and vice versa, the **bilinear transformation** is used, which transforms stable systems in the S-domain to stable systems in the Z-domain (Section 4.2.1). Using the bilinear transformation, a predistortion of the $\Omega$-axis in the Z-domain is necessary. The design may be done using the standard procedures described in Section 4.2.2 and 4.2.3 using an auxiliary function.

For the lowpass-design, the functions provided by MATLAB are represented in the following table. Calculation of the necessary filter order and the normalized cut-off frequency\footnote{All design functions use normalized frequencies between 0 and 1. Normalization is done according to half of the sampling frequency. The normalized cut-off frequency indicates the frequency with $3 \, \text{dB}$ attenuation. Example: Sampling frequency 1000 Hz, cut-off frequency 300 Hz, normalized cut-off frequency: `Wn=300/500=0.6`} can be done using these functions:

| Filter Type        | Determination of Filter Order           |
| ------------------ | --------------------------------------- |
| Butterworth        | `[m, Wn] = buttord(Wp, Ws, Rp, Rs)`     |
| Chebyshev type I   | `[m, Wn] = cheb1ord(Wp, Ws, Rp, Rs)`    |
| Chebyshev type II  | `[m, Wn] = cheb2ord(Wp, Ws, Rp, Rs)`    |
| Cauer              | `[m, Wn] = ellipord(Wp, Ws, Rp, Rs)`    |

Subsequently, the coefficients can be calculated using the design functions:

| Filter Type        | Design Function                         |
| ------------------ | --------------------------------------- |
| Butterworth        | `[b, a] = butter(m, Wn)`                |
| Chebyshev type I   | `[b, a] = cheby1(m, Rp, Wn)`            |
| Chebyshev type II  | `[b, a] = cheby2(m, Rs, Wn)`            |
| Cauer              | `[b, a] = ellip(m, Rp, Rs, Wn)`         |

The meaning of the parameters are:

| Parameter          | Description                                        |
| ------------------ | -------------------------------------------------- |
| `b, a`             | vectors with the filter coefficients               |
| `m`                | order of the filter                                |
| `Wn`               | normalized cut-off frequency of the lowpass        |
| `Wp`               | normalized pass-band frequency of the lowpass      |
| `Ws`               | normalized stop-band frequency of the lowpass      |
| `Rp`               | ripple in pass-band (dB)                           |
| `Rs`               | stop-band attenuation (dB)                         |

Determine the minimal filter order needed for each filter type and design the lowpass-filters for the following tolerance scheme:

- $f_\text{D} = 1 \, \text{kHz}$
- $f_\text{S} = 1.4 \, \text{kHz}$
- $R_\text{P} = 0.5 \, \text{dB}$
- $R_\text{S} = 30 \, \text{dB}$
- $f_A = 8 \, \text{kHz}$

![Tolerance Scheme](./pngs/toleranz.eps)

Compare the amplitude response and the pole-zero plot of the different filters to each other.

In [ ]:
# BEGIN SOLUTION 
fD = 1000;    % Passband frequency (Hz)
fS = 1400;    % Stopband frequency (Hz)
Rp = 0.5;     % Passband ripple (dB)
Rs = 30;      % Stopband attenuation (dB)
fA = 8000;    % Sampling frequency (Hz)
Wp = 2*fD/fA; % Conversion to normalized cutoff frequencies
Ws = 2*fS/fA;

for typ = 1:4
    if typ == 1
        [n, Wn] = buttord(Wp, Ws, Rp, Rs);  % Butterworth
        [b, a] = butter(n, Wn);
        FType = 'Butterworth';
    elseif typ == 2
        [n, Wn] = cheb1ord(Wp, Ws, Rp, Rs); % Chebyshev Type I
        [b, a] = cheby1(n, Rp, Wn);
        FType = 'Chebyshev Type I';
    elseif typ == 3
        [n, Wn] = cheb2ord(Wp, Ws, Rp, Rs); % Chebyshev Type II
        [b, a] = cheby2(n, Rs, Wn);
        FType = 'Chebyshev Type II';
    elseif typ == 4
        [n, Wn] = ellipord(Wp, Ws, Rp, Rs); % Cauer
        [b, a] = ellip(n, Rp, Rs, Wn);
        FType = 'Cauer';
    else
        error('Something went wrong...');
    end

    [H, W] = freqz(b, a, 512);
    
    % # Graphical Output # %
    TitleString = sprintf('Filter Type: %s  Order: %d', FType, n);
    
    % Frequency Response
    figure; 
    plot(W/pi, abs(H)); 
    grid; 
    title(TitleString);
    xlabel('\Omega/\pi');
    ylabel('|H(exp(j\Omega))|'); 
    axis([0 1 0 1.2]);
    
    % Drawing the tolerance band
    xline = [Wp Wp];
    yline = [0 10.^(-Rp/20)];           
    line(xline, yline, 'linestyle', '--');
    xline = [0 Ws]; 
    yline = [1 1];                      
    line(xline, yline, 'linestyle', '--');
    xline = [0 Wp]; 
    yline = [10.^(-Rp/20) 10.^(-Rp/20)];
    line(xline, yline, 'linestyle', '--');
    xline = [Ws Ws];
    yline = [10.^(-Rs/20) 1];
    line(xline, yline, 'linestyle', '--');
    xline = [Ws 1]; 
    yline = [10.^(-Rs/20) 10.^(-Rs/20)];
    line(xline, yline, 'linestyle', '--');
    
    % Pole and Zero Plot in the z-Plane
    figure; 
    zplane(b, a); 
    title(TitleString); 
    xlabel('Real Part');
    ylabel('Imaginary Part');
end
# END SOLUTION